# 02b — Temperatura de superfície (LST) e NDVI — Landsat 8/9 via Microsoft Planetary Computer
**Alternativa ao Earth Engine, sem cadastro e sem faturamento.** Mesmo dado de origem: Landsat Collection 2 Nível 2
(banda termal `lwir11` = ST_B10), lido direto do catálogo STAC aberto do Planetary Computer.

Gera os mesmos arquivos do notebook 02, então os notebooks 03–06 rodam sem mudança:
1. `data/raw/lst/lst_composto.tif`: LST mín/média/mediana/p90/máx/desvio, anomalia média/p90 e NDVI mediano (30 m, EPSG:31983)
2. `data/raw/lst/lst_serie_escolas.csv`: LST e anomalia cena a cena por escola (média ponderada por fração de pixel, raio de 100 m)

Use **Ambiente de execução › Alterar tipo › RAM alta** se disponível. Tempo estimado: 15–40 min.

In [ ]:
!pip -q install pystac-client planetary-computer odc-stac rasterio exactextract geopandas

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import pystac_client, planetary_computer
from odc.stac import load as stac_load
from rasterio.transform import from_bounds

INI, FIM = '2019-01-01', '2025-12-31'
MESES = [2, 3, 4, 10, 11, 12]     # meses letivos mais quentes
MAX_NUVEM = 40                     # % de nuvem da cena
MIN_VALIDO = 0.6                   # fração mínima de pixels válidos na área
MAX_CENAS = 90                     # teto de memória (as de menor nuvem primeiro)

esc = gpd.read_file(f'{INT}/escolas.gpkg', layer='escolas')
area = gpd.GeoSeries([esc.unary_union.convex_hull.buffer(3000)], crs=CRS)
bbox_ll = tuple(area.to_crs(4326).total_bounds)
xmin, ymin, xmax, ymax = area.total_bounds

cat = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                modifier=planetary_computer.sign_inplace)
itens = cat.search(collections=['landsat-c2-l2'], bbox=bbox_ll, datetime=f'{INI}/{FIM}',
                   query={'eo:cloud_cover': {'lt': MAX_NUVEM},
                          'platform': {'in': ['landsat-8', 'landsat-9']}}).item_collection()
itens = [i for i in itens if i.datetime.month in MESES]
itens = sorted(itens, key=lambda i: i.properties['eo:cloud_cover'])
print(len(itens), 'cenas candidatas')

In [ ]:
# lê cena a cena (recorte da área, 30 m, EPSG:31983) e aplica a máscara de nuvem
def le_cena(item):
    ds = stac_load([item], bands=['lwir11', 'red', 'nir08', 'qa_pixel'], crs=CRS, resolution=30,
                   x=(xmin, xmax), y=(ymin, ymax)).isel(time=0)
    qa = ds['qa_pixel'].values.astype(np.uint16)
    limpo = ((qa & (1 << 1)) == 0) & ((qa & (1 << 3)) == 0) & ((qa & (1 << 4)) == 0) & ((qa & (1 << 5)) == 0) & (qa != 1)
    st = ds['lwir11'].values.astype(np.float32)
    lst = np.where(limpo & (st > 0), st * 0.00341802 + 149.0 - 273.15, np.nan).astype(np.float32)
    red = ds['red'].values * 0.0000275 - 0.2; nir = ds['nir08'].values * 0.0000275 - 0.2
    ndvi = np.where(limpo, (nir - red) / (nir + red + 1e-6), np.nan).astype(np.float32)
    return lst, ndvi, ds

LST, NDVI, datas = [], [], []
ref = None
for _k, it in enumerate(itens):
    if len(LST) >= MAX_CENAS: break
    if it.datetime.strftime('%Y-%m-%d') in datas: continue   # mesma data (cenas sobrepostas)
    try:
        lst, ndvi, ds = le_cena(it)
    except Exception as e:
        print('falhou', it.id, e); continue
    frac = np.isfinite(lst).mean()
    if frac < MIN_VALIDO: continue
    if ref is None: ref = ds   # grade de referência
    LST.append(lst); NDVI.append(ndvi); datas.append(it.datetime.strftime('%Y-%m-%d'))
    print(f'{len(LST):3d} {datas[-1]} válido={frac:.2f}')
LST = np.stack(LST); NDVI = np.stack(NDVI)
print(LST.shape)

In [ ]:
# compostos
media_cena = np.nanmean(LST, axis=(1, 2))[:, None, None]
ANOM = LST - media_cena
comp = {
    'LST_min': np.nanmin(LST, 0), 'LST_mean': np.nanmean(LST, 0), 'LST_median': np.nanmedian(LST, 0),
    'LST_p90': np.nanpercentile(LST, 90, axis=0), 'LST_max': np.nanmax(LST, 0), 'LST_stdDev': np.nanstd(LST, 0),
    'ANOM_mean': np.nanmean(ANOM, 0), 'ANOM_p90': np.nanpercentile(ANOM, 90, axis=0),
    'NDVI_median': np.nanmedian(NDVI, 0),
}
x = ref['x'].values; y = ref['y'].values
transf = from_bounds(x.min() - 15, y.min() - 15, x.max() + 15, y.max() + 15, len(x), len(y))
os.makedirs(f'{RAW}/lst', exist_ok=True)
perfil = dict(driver='GTiff', height=len(y), width=len(x), dtype='float32', crs=CRS, transform=transf,
              nodata=np.nan, compress='deflate', tiled=True)
with rasterio.open(f'{RAW}/lst/lst_composto.tif', 'w', count=len(comp), **perfil) as dst:
    for i, (nome, arr) in enumerate(comp.items(), 1):
        dst.write(arr.astype(np.float32), i); dst.set_band_description(i, nome)
print('composto salvo; cenas usadas:', len(datas))

In [ ]:
# série temporal por escola: pilha de cenas → exactextract (média ponderada por fração de pixel)
from exactextract import exact_extract
pilha = '/content/lst_cenas.tif'   # disco local do Colab (não vai para o Drive)
with rasterio.open(pilha, 'w', count=len(datas), **perfil) as dst:
    for i, d in enumerate(datas, 1):
        dst.write(LST[i - 1], i); dst.set_band_description(i, d)
buf = gpd.read_file(f'{INT}/escolas.gpkg', layer=f'buffer_{BUFFER_REF}m')
z = exact_extract(pilha, buf, 'mean', include_cols=['cod_esc'], output='pandas')
z.columns = ['cod_esc'] + datas if len(z.columns) == len(datas) + 1 else z.columns
serie = z.melt(id_vars='cod_esc', var_name='data', value_name='LST')
serie['data'] = serie['data'].str.replace('_mean', '', regex=False)
mc = pd.Series(media_cena.ravel(), index=datas)
serie['ANOM'] = serie['LST'] - serie['data'].map(mc)
serie.to_csv(f'{RAW}/lst/lst_serie_escolas.csv', index=False)
print(serie.describe())

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(comp['ANOM_mean'], cmap='RdYlBu_r', vmin=-5, vmax=5); ax[0].set_title('Anomalia média de LST (°C)')
ax[1].imshow(comp['NDVI_median'], cmap='Greens', vmin=0, vmax=0.8); ax[1].set_title('NDVI mediano')
for a in ax: a.set_axis_off()
plt.savefig(f'{OUT}/lst_anomalia_ndvi.png', dpi=150, bbox_inches='tight')